# Step 2: Deploy Lake Formation RLS

Set up enterprise-grade row-level security using AWS Lake Formation.

## Prerequisites

- ✅ Run `01-deploy-athena.ipynb` first
- ✅ Athena database and tables created

## What This Notebook Does

1. Creates IAM role for Lake Formation RLS
2. Configures data filters on claims table
3. Sets up row-level security: `user_id = '${aws:PrincipalTag/user_id}'`
4. Saves RLS_ROLE_ARN to SSM

## Next Notebook

- **03-deploy-cognito.ipynb** - Set up user authentication

In [ ]:
import boto3
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from config import config

lakeformation_client = boto3.client('lakeformation')
iam_client = boto3.client('iam')
ssm_client = boto3.client('ssm')

print("✅ Setup complete")

## Step 1: Run Lake Formation Setup

In [ ]:
import subprocess

result = subprocess.run(
    ['python', 'setup_lake_formation.py', '--bucket', config.S3_BUCKET_NAME],
    cwd='athena-setup',
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Error:", result.stderr)
else:
    print("\n✅ Lake Formation RLS setup complete!")
    
    # Extract RLS_ROLE_ARN from output
    for line in result.stdout.split('\n'):
        if 'RLS_ROLE_ARN' in line or 'Role ARN' in line:
            print(f"\n📋 {line}")

## Step 2: Save RLS Role ARN to SSM

**IMPORTANT**: Copy the RLS_ROLE_ARN from above and paste it below.

In [ ]:
# UPDATE THIS with the actual ARN from Step 1
RLS_ROLE_ARN = 'arn:aws:iam::ACCOUNT_ID:role/lakehouse-rls-role'  # CHANGE THIS

# Save to SSM
ssm_client.put_parameter(
    Name='lh_rls_role_arn',
    Value=RLS_ROLE_ARN,
    Type='String',
    Description='Lake Formation RLS role ARN',
    Overwrite=True
)

print(f"✅ Saved lh_rls_role_arn to SSM")
print(f"   Value: {RLS_ROLE_ARN}")

## Summary

✅ **Lake Formation RLS Deployment Complete!**

**What was created:**
- IAM role: `lakehouse-rls-role`
- Data filter: `user_id = '${aws:PrincipalTag/user_id}'`
- Lake Formation permissions configured

**Next Steps:**
Run **03-deploy-cognito.ipynb** to set up user authentication